# Offline run analysis

Set `RUN_ID` to an existing online run folder for raw input data, and set `OFFLINE_RUN_ID` for this offline analysis pass. This notebook loads raw training and realtime trial `.npz` files from `runs/<RUN_ID>/`, applies its own preprocessing and audio-cue labeling settings, trains offline model variants, tests each on the realtime trial, and saves all notebook-03 outputs under `runs_offline/<OFFLINE_RUN_ID>/`. Models use time-domain EEG windows after this notebook's preprocessing step.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'training.py').exists():
    ROOT = Path(r'D:/BME/BCI/online_bci/online_eeg')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from preprocessing import (
    AudioLabelConfig,
    PreprocessConfig,
    labeled_preprocess_summary,
    preprocess_recording,
)
from training import TrainingConfig, offline_train_test_sweep
from plots import plot_labeled_recording, plot_predictions_overlay

print('Pipeline root:', ROOT)


## Raw files, preprocessing, and sweep settings


In [ ]:
RUN_ID = 'run_001'
RUN_DIR = ROOT / 'runs' / RUN_ID

OFFLINE_RUN_ID = 'offline_001'  # Change this to keep a separate set of 03 outputs.
OFFLINE_ROOT = ROOT / 'runs_offline'
OFFLINE_RUN_DIR = OFFLINE_ROOT / OFFLINE_RUN_ID

TRAIN_RAW_NPZ = RUN_DIR / 'raw_training' / 'train_01.npz'
TEST_RAW_NPZ = RUN_DIR / 'realtime_trials' / 'realtime_trial_01_raw.npz'

EEG_CHANNELS = (1, 2, 3, 4)
EEG_CHANNEL_NAMES = ('O1', 'Oz', 'O2', 'POz')
AUDIO_CHANNEL = 16

APPLY_SOFTWARE_FILTERS = True  # BIOPAC hardware already bandpasses the EEG at 1-35 Hz.
DEMEAN_CHANNELS = True
SOFTWARE_BANDPASS_HZ = (8.0, 35.0)  # Set to (8.0, 35.0) to filter out blinks only if APPLY_SOFTWARE_FILTERS=True.
SOFTWARE_NOTCH_HZ = (60.0,)  # Set to (60.0,) only if APPLY_SOFTWARE_FILTERS=True.
PREPROCESS_TAG = 'software_filters_on' if APPLY_SOFTWARE_FILTERS else 'hardware_filter_only'
if DEMEAN_CHANNELS:
    PREPROCESS_TAG += '_demeaned'

OFFLINE_LABELED_DIR = OFFLINE_RUN_DIR / 'labeled'
TRAIN_LABELED_NPZ = OFFLINE_LABELED_DIR / f'{TRAIN_RAW_NPZ.stem}_labeled.npz'
TEST_LABELED_NPZ = OFFLINE_LABELED_DIR / f'{TEST_RAW_NPZ.stem}_labeled.npz'
SWEEP_DIR = OFFLINE_RUN_DIR / 'sweeps'

FEATURE_MODES = ('filtered_signal',)

# WINDOW_SECS = (1.0, 2.0, 3.0)
WINDOW_SECS = (0.5, 1.0, 1.5, 2.0)
# STRIDE_SECS = (0.05, 0.1, 0.2)
STRIDE_SECS = (0.05, 0.1, 0.15, 0.2)
LABEL_MODES = ('endpoint', 'majority')  # Include one or both: 'endpoint', 'majority'.

PRE = PreprocessConfig(
    eeg_channels=EEG_CHANNELS,
    audio_channel=AUDIO_CHANNEL,
    apply_software_filters=APPLY_SOFTWARE_FILTERS,
    bandpass_low_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[0],
    bandpass_high_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[1],
    notch_hz=SOFTWARE_NOTCH_HZ,
    notch_quality_factor=30.0,
    filter_order=4,
    demean_channels=DEMEAN_CHANNELS,
)

LABELS = AudioLabelConfig(
    class_names=('Eyes Open', 'Eyes Closed'),
    baseline_label=0,
    active_label=1,
    cue_label_sequence=None,
    alternate_binary_labels=True,
    label_duration_sec=None,  # transition mode: each cue switches state until the next cue.
    label_start_offset_sec=0.0,  # label switch starts exactly at cue onset.
    envelope_window_sec=0.025,
    onset_threshold=None,
    onset_min_interval_sec=0.50,
)

TRAIN = TrainingConfig(
    train_fraction=0.9,
    hidden_size=64,
    num_layers=2,
    dropout=0.25,
    batch_size=64,
    epochs=20,
    lr=1e-3,
    seed=888,
)

if not RUN_DIR.exists():
    raise FileNotFoundError(f'Run folder not found: {RUN_DIR}')
if not TRAIN_RAW_NPZ.exists():
    raise FileNotFoundError(f'Training raw file not found: {TRAIN_RAW_NPZ}')
if not TEST_RAW_NPZ.exists():
    raise FileNotFoundError(f'Test raw file not found: {TEST_RAW_NPZ}')

OFFLINE_LABELED_DIR.mkdir(parents=True, exist_ok=True)
SWEEP_DIR.mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_DIR)
print('Offline root:', OFFLINE_ROOT)
print('Offline run directory:', OFFLINE_RUN_DIR)
print('Training raw file:', TRAIN_RAW_NPZ)
print('Test raw file:', TEST_RAW_NPZ)
print('Offline labeled output:', OFFLINE_LABELED_DIR)
print('Sweep output:', SWEEP_DIR)
print('Preprocess tag:', PREPROCESS_TAG)
print('Software filters enabled:', APPLY_SOFTWARE_FILTERS)


## Preprocess raw files for this offline analysis


In [ ]:
TRAIN_LABELED_NPZ, train_cue_table = preprocess_recording(
    raw_npz=TRAIN_RAW_NPZ,
    output_npz=TRAIN_LABELED_NPZ,
    preprocess_config=PRE,
    label_config=LABELS,
)
TEST_LABELED_NPZ, test_cue_table = preprocess_recording(
    raw_npz=TEST_RAW_NPZ,
    output_npz=TEST_LABELED_NPZ,
    preprocess_config=PRE,
    label_config=LABELS,
)

preprocess_summary = labeled_preprocess_summary({
    'training': TRAIN_LABELED_NPZ,
    'realtime_test': TEST_LABELED_NPZ,
})
display(preprocess_summary)

print('Training cue table')
display(train_cue_table)
print('Realtime test cue table')
display(test_cue_table)


## Inspect freshly labeled train/test data


In [ ]:
fig, axes = plot_labeled_recording(
    TRAIN_LABELED_NPZ,
    max_duration_sec=30.0,
    channel_names=EEG_CHANNEL_NAMES,
)
axes[0].set_title('Training labeled EEG preview')


In [ ]:
fig, axes = plot_labeled_recording(
    TEST_LABELED_NPZ,
    max_duration_sec=None,
    channel_names=EEG_CHANNEL_NAMES,
)
axes[0].set_title('Realtime trial labeled EEG preview')


## Run offline model sweep


In [ ]:
sweep_result = offline_train_test_sweep(
    train_labeled_npz=TRAIN_LABELED_NPZ,
    test_labeled_npz=TEST_LABELED_NPZ,
    output_dir=SWEEP_DIR,
    feature_modes=FEATURE_MODES,
    window_secs=WINDOW_SECS,
    stride_secs=STRIDE_SECS,
    training_config=TRAIN,
    label_modes=LABEL_MODES,
)

RANK_COLUMN = 'test_xcov_peak_coeff'
summary = sweep_result['summary'].copy()
if RANK_COLUMN not in summary.columns:
    raise KeyError(f'Missing ranking column: {RANK_COLUMN}')
summary = (
    summary
    .sort_values(
        [RANK_COLUMN, 'test_balanced_accuracy', 'val_balanced_accuracy'],
        ascending=[False, False, False],
        na_position='last',
    )
    .reset_index(drop=True)
)
print('Saved sweep summary:', sweep_result['summary_csv'])
print(f'Ranked variants by {RANK_COLUMN} descending; balanced accuracy is used only as a tie-breaker.')
display(summary)


## Compare variants


In [ ]:
TOP_N_VARIANTS = 10
plot_df = summary.head(TOP_N_VARIANTS).copy()
plot_df['variant_short'] = plot_df['variant'].str.replace('__', '\n', regex=False)
ax = plot_df.plot.bar(
    x='variant_short',
    y=['val_balanced_accuracy', 'test_balanced_accuracy'],
    figsize=(35, 15),
)
ax.set_xlabel('Variant', fontsize=30, labelpad=16)
ax.set_ylabel('Balanced accuracy', fontsize=30, labelpad=16)
ax.set_title(
    f'Top {TOP_N_VARIANTS} offline sweep results for {RUN_ID} ranked by xcov peak coefficient',
    fontsize=38,
    pad=24,
)
ax.tick_params(axis='x', labelsize=27, rotation=90)
ax.tick_params(axis='y', labelsize=27)
ax.grid(True, axis='y', alpha=0.3)

ax2 = ax.twinx()
xcov_values = pd.to_numeric(plot_df['test_xcov_peak_coeff'], errors='coerce')
line = ax2.plot(
    range(len(plot_df)),
    xcov_values,
    color='black',
    marker='o',
    linewidth=2.5,
    markersize=8,
    label='test_xcov_peak_coeff',
)
ax2.set_ylabel('Normalized xcov peak coefficient', fontsize=30, labelpad=16)
ax2.tick_params(axis='y', labelsize=27)
if xcov_values.notna().any():
    y_min = min(0.0, float(xcov_values.min()))
    y_max = max(1.0, float(xcov_values.max()))
    pad = max(0.05, 0.08 * (y_max - y_min))
    ax2.set_ylim(y_min - pad, y_max + pad)
    for idx, value in enumerate(xcov_values):
        if pd.notna(value):
            ax2.annotate(
                f'{float(value):.2f}',
                xy=(idx, float(value)),
                xytext=(0, 8),
                textcoords='offset points',
                ha='center',
                va='bottom',
                fontsize=20,
                color='black',
            )

handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax.legend(handles1 + handles2, labels1 + labels2, fontsize=25, loc='best')
plt.tight_layout()


## Inspect best variant predictions


In [ ]:
best = summary.iloc[0]
best_variant_dir = Path(best['variant_dir'])
best_pred_csv = best_variant_dir / f'{TEST_LABELED_NPZ.stem}_test_predictions.csv'
best_aligned_pred_csv = Path(best.get('test_aligned_prediction_csv', best_variant_dir / f'{TEST_LABELED_NPZ.stem}_test_predictions_aligned_eeg.csv'))
best_cue_delay_summary_csv = best_variant_dir / f'{TEST_LABELED_NPZ.stem}_test_cue_delay_summary.csv'
best_xcov_delay_summary_csv = best_variant_dir / f'{TEST_LABELED_NPZ.stem}_test_xcov_delay_summary.csv'
best_xcov_curve_csv = best_variant_dir / f'{TEST_LABELED_NPZ.stem}_test_xcov_curve.csv'

print('Best variant by xcov peak coefficient:', best['variant'])
print('Best xcov peak coefficient:', best.get('test_xcov_peak_coeff'))
print('Best xcov delay sec:', best.get('test_xcov_delay_sec'))
print('Best predictions:', best_pred_csv)
print('Best aligned EEG/predictions:', best_aligned_pred_csv)

best_predictions = pd.read_csv(best_pred_csv)
display(best_predictions.tail())

best_cue_delay_summary = pd.read_csv(best_cue_delay_summary_csv) if best_cue_delay_summary_csv.exists() else None
best_xcov_delay_summary = pd.read_csv(best_xcov_delay_summary_csv) if best_xcov_delay_summary_csv.exists() else None
best_xcov_curve = pd.read_csv(best_xcov_curve_csv) if best_xcov_curve_csv.exists() else None

if best_cue_delay_summary is not None:
    display(best_cue_delay_summary)
if best_xcov_delay_summary is not None:
    display(best_xcov_delay_summary)

fig, axes = plot_predictions_overlay(
    TEST_LABELED_NPZ,
    best_predictions,
    max_duration_sec=None,
    channel_names=EEG_CHANNEL_NAMES,
    show_true_labels=True,
)
axes[0].set_title(f"Best offline variant by xcov peak coefficient on realtime trial: {best['variant']}")

if best_xcov_curve is not None and best_xcov_delay_summary is not None:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(best_xcov_curve['lag_sec'], best_xcov_curve['xcov_coeff'], linewidth=1.8)
    peak_delay = float(best_xcov_delay_summary.loc[0, 'xcov_delay_sec'])
    peak_coeff = float(best_xcov_delay_summary.loc[0, 'xcov_peak_coeff'])
    ax.axvline(peak_delay, linestyle=':', color='tab:red', linewidth=2.0, label=f'peak lag = {peak_delay:.3f} s')
    ax.scatter([peak_delay], [peak_coeff], color='tab:red', zorder=3)
    ax.axvline(0.0, linestyle='--', color='black', alpha=0.5, linewidth=1.0)
    ax.set_xlabel('Lag (s)', fontsize=14)
    ax.set_ylabel('Normalized xcov coefficient', fontsize=14)
    ax.set_title(f"Prediction xcov lag for best xcov-ranked variant: {best['variant']}", fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12)
    plt.tight_layout()
